<a href="https://colab.research.google.com/github/mrdantzig/nlp-bible-code/blob/master/24%EC%9E%A5_%EB%94%A5%EB%9F%AC%EB%8B%9D%20%EA%B8%B0%EB%B0%98%20Question%20%26%20%20Answering/%5B24-1%5DBERT%EB%A5%BC%20%ED%99%9C%EC%9A%A9%ED%95%9C%20KorQuAD%20%EB%8D%B0%EC%9D%B4%ED%84%B0%EC%85%8B%20%ED%95%99%EC%8A%B5%20%EB%B0%8F%20%ED%8F%89%EA%B0%80_%EC%88%98%EC%A0%95.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [24-1] BERT를 활용한 KorQuAD 데이터셋 학습 및 평가 (수정본)

자연어처리 바이블 24장 실습 [24-1]을 최신 Colab에서 실행되도록 고친 노트북입니다. 다국어 BERT를 한국어 기계독해 데이터셋 **KorQuAD 1.0**으로 미세조정해, 문맥(context)에서 질문의 정답 구간을 찾습니다.

**원본 대비 수정 사항**
1. **CLaF → Hugging Face Transformers**: 원본의 Naver CLaF(2019)는 Python 3.6·PyTorch 1.x·TensorFlow 1 기준이라 최신 Colab에서 설치되지 않습니다. 같은 작업(BERT 기반 추출형 QA 학습·평가·예측)을 `transformers`의 표준 코드로 바꿨습니다.
2. **모델: `bert-base-multilingual-uncased` → `bert-base-multilingual-cased`**: uncased 모델은 전처리에서 한글 음절을 자모로 분해해 한국어 성능이 떨어집니다. Google도 다국어 모델은 cased 사용을 권장합니다.
3. **불필요한 단계 삭제**: Google Drive 마운트, `cd /root`, mecab 설치는 필요 없습니다.
4. **학습 시간 조절**: 기본값은 학습 데이터 60,407개 중 20,000개 · 1 epoch(Colab T4 GPU 약 10분)입니다. `MAX_TRAIN = None`으로 바꾸면 전체를 학습합니다.
5. **평가 지표 직접 계산**: KorQuAD 공식 평가 방식대로 **EM(완전 일치)**과 **음절 단위 F1**을 계산합니다.
6. **예측**: 원본의 예시("88올림픽 개최지는?")를 그대로 사용합니다.

※ **런타임 → 런타임 유형 변경 → T4 GPU**를 선택한 뒤 실행하세요.

## 1. 환경 설정

In [ ]:
# Colab에 기본 설치된 버전 사용(-U로 업그레이드하면 pyarrow 등이 바뀌어 의존성 충돌 경고가 납니다)
!pip install -q transformers datasets accelerate

import collections
import json
import re
import string
import urllib.request

import numpy as np
import torch
import transformers
from datasets import Dataset
from transformers import (AutoModelForQuestionAnswering, AutoTokenizer, Trainer,
                          TrainingArguments, default_data_collator)

print("transformers", transformers.__version__, "| GPU:", torch.cuda.is_available())

## 2. KorQuAD 1.0 데이터 내려받기

KorQuAD는 SQuAD와 같은 형식입니다: 문서(`title`) → 문단(`context`) → 질문·정답(`question`, `answers.text`, `answers.answer_start`).

In [ ]:
BASE = "https://korquad.github.io/dataset/"
FILES = {"train": "KorQuAD_v1.0_train.json", "dev": "KorQuAD_v1.0_dev.json"}


def load_korquad(path):
    data = json.load(open(path, encoding="utf-8"))["data"]
    rows = []
    for article in data:
        for para in article["paragraphs"]:
            for qa in para["qas"]:
                rows.append({"id": qa["id"], "title": article.get("title", ""),
                             "context": para["context"], "question": qa["question"],
                             "answers": {"text": [a["text"] for a in qa["answers"]],
                                         "answer_start": [a["answer_start"] for a in qa["answers"]]}})
    return Dataset.from_list(rows)


try:
    for f in FILES.values():
        urllib.request.urlretrieve(BASE + f, f)
    train_raw = load_korquad(FILES["train"])
    dev_raw = load_korquad(FILES["dev"])
except Exception as e:                                   # 공식 사이트 접속이 안 될 때: Hugging Face 사본
    print("공식 사이트 다운로드 실패 → Hugging Face 데이터셋 사용:", e)
    from datasets import load_dataset
    ds = load_dataset("KorQuAD/squad_kor_v1")
    train_raw, dev_raw = ds["train"], ds["validation"]

print("train:", len(train_raw), "/ dev:", len(dev_raw))   # 60407 / 5774
ex = dev_raw[0]
print("Q:", ex["question"], "\nA:", ex["answers"]["text"], "\nC:", ex["context"][:150], "...")

## 3. 토크나이저와 모델

- 문맥이 길면 `max_length=384`, `stride=128`로 겹치게 잘라 여러 조각(feature)으로 만듭니다.
- 학습 시에는 각 조각에서 정답의 **시작·끝 토큰 위치**를 정답 레이블로 만들고, 정답이 조각 밖이면 [CLS] 위치(0)를 정답으로 둡니다.

In [ ]:
MODEL_NAME = "bert-base-multilingual-cased"   # 수정: uncased → cased (한글 자모 분해 방지)
MAX_LEN, STRIDE = 384, 128
MAX_TRAIN = 20000                             # None이면 전체(60,407개) 학습

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)
print(tokenizer.tokenize("1988년 서울 올림픽은 9월 17일에 개최되었다."))


def prepare_train_features(examples):
    enc = tokenizer([q.strip() for q in examples["question"]], examples["context"],
                    truncation="only_second", max_length=MAX_LEN, stride=STRIDE,
                    return_overflowing_tokens=True, return_offsets_mapping=True, padding="max_length")
    sample_map = enc.pop("overflow_to_sample_mapping")
    offsets_all = enc.pop("offset_mapping")
    enc["start_positions"], enc["end_positions"] = [], []
    for i, offsets in enumerate(offsets_all):
        seq_ids = enc.sequence_ids(i)
        ans = examples["answers"][sample_map[i]]
        s_char = ans["answer_start"][0]
        e_char = s_char + len(ans["text"][0])
        ctx_s = seq_ids.index(1)
        ctx_e = len(seq_ids) - 1 - seq_ids[::-1].index(1)
        if offsets[ctx_s][0] > s_char or offsets[ctx_e][1] < e_char:   # 정답이 이 조각 밖
            enc["start_positions"].append(0)
            enc["end_positions"].append(0)
        else:
            ts = ctx_s
            while ts <= ctx_e and offsets[ts][0] <= s_char:
                ts += 1
            te = ctx_e
            while te >= ctx_s and offsets[te][1] >= e_char:
                te -= 1
            enc["start_positions"].append(ts - 1)
            enc["end_positions"].append(te + 1)
    return enc


def prepare_eval_features(examples):
    enc = tokenizer([q.strip() for q in examples["question"]], examples["context"],
                    truncation="only_second", max_length=MAX_LEN, stride=STRIDE,
                    return_overflowing_tokens=True, return_offsets_mapping=True, padding="max_length")
    sample_map = enc.pop("overflow_to_sample_mapping")
    enc["example_id"] = []
    for i in range(len(enc["input_ids"])):
        seq_ids = enc.sequence_ids(i)
        enc["example_id"].append(examples["id"][sample_map[i]])
        # 문맥이 아닌 토큰의 offset은 None으로 → 후처리에서 정답 후보에서 제외
        enc["offset_mapping"][i] = [o if seq_ids[k] == 1 else None
                                    for k, o in enumerate(enc["offset_mapping"][i])]
    return enc


train_src = train_raw.shuffle(seed=42).select(range(MAX_TRAIN)) if MAX_TRAIN else train_raw
train_ds = train_src.map(prepare_train_features, batched=True, remove_columns=train_raw.column_names)
dev_feats = dev_raw.map(prepare_eval_features, batched=True, remove_columns=dev_raw.column_names)
print("학습 feature:", len(train_ds), "/ 평가 feature:", len(dev_feats))

## 4. 학습

원본 CLaF 설정(batch 4)보다 큰 batch 16, learning rate 3e-5를 씁니다. GPU에서는 fp16으로 속도를 높입니다.

In [ ]:
args = TrainingArguments(
    output_dir="korquad-mbert",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=3e-5,
    num_train_epochs=1,
    weight_decay=0.01,
    warmup_steps=100,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    save_strategy="no",
    report_to="none",
    seed=42,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  data_collator=default_data_collator)
trainer.train()

## 5. 평가 — EM과 F1 (KorQuAD 공식 방식)

1. 각 조각에서 시작·끝 로짓 상위 20개씩을 조합해 가장 점수가 높은 구간(최대 30토큰)을 고릅니다.
2. **EM**: 정답과 완전히 같으면 1, **F1**: 한국어는 조사가 붙는 경우가 많아 **음절(글자) 단위**로 겹치는 정도를 봅니다.

In [ ]:
def postprocess(examples, features, start_logits, end_logits, n_best=20, max_answer_len=30):
    feats_per_ex = collections.defaultdict(list)
    for i, ex_id in enumerate(features["example_id"]):
        feats_per_ex[ex_id].append(i)
    preds = {}
    for ex in examples:
        best, best_score = "", -1e9
        for fi in feats_per_ex[ex["id"]]:
            offsets = features[fi]["offset_mapping"]
            s_idx = np.argsort(start_logits[fi])[-1:-n_best - 1:-1]
            e_idx = np.argsort(end_logits[fi])[-1:-n_best - 1:-1]
            for s in s_idx:
                for e in e_idx:
                    if offsets[s] is None or offsets[e] is None or e < s or e - s + 1 > max_answer_len:
                        continue
                    score = start_logits[fi][s] + end_logits[fi][e]
                    if score > best_score:
                        best_score, best = score, ex["context"][offsets[s][0]:offsets[e][1]]
        preds[ex["id"]] = best
    return preds


# KorQuAD 1.0 공식 평가 스크립트와 같은 정규화 · 음절 단위 F1
def normalize_answer(s):
    s = re.sub(r"""'|"|《|》|<|>|〈|〉|\(|\)|‘|’""", " ", s)
    s = "".join(ch for ch in s.lower() if ch not in set(string.punctuation))
    return " ".join(s.split())


def f1_score(pred, gt):
    p = [c for tok in normalize_answer(pred).split() for c in tok]
    g = [c for tok in normalize_answer(gt).split() for c in tok]
    common = collections.Counter(p) & collections.Counter(g)
    same = sum(common.values())
    if same == 0:
        return 0.0
    precision, recall = same / len(p), same / len(g)
    return 2 * precision * recall / (precision + recall)


def exact_match(pred, gt):
    return float(normalize_answer(pred) == normalize_answer(gt))


def evaluate_korquad(examples, preds):
    em = f1 = 0.0
    for ex in examples:
        gts = ex["answers"]["text"]
        em += max(exact_match(preds[ex["id"]], g) for g in gts)
        f1 += max(f1_score(preds[ex["id"]], g) for g in gts)
    return {"exact_match": 100 * em / len(examples), "f1": 100 * f1 / len(examples)}


out = trainer.predict(dev_feats.remove_columns(["example_id", "offset_mapping"]))
start_logits, end_logits = out.predictions
dev_preds = postprocess(dev_raw, dev_feats, start_logits, end_logits)
print(evaluate_korquad(dev_raw, dev_preds))

for ex in dev_raw.select(range(5)):
    print(f"Q: {ex['question']}\n   정답: {ex['answers']['text'][0]} | 예측: {dev_preds[ex['id']]}")

## 6. 학습된 모델로 예측하기 (원본 예시)

In [ ]:
def answer(question, context):
    ex = Dataset.from_list([{"id": "q", "question": question, "context": context,
                             "answers": {"text": [""], "answer_start": [0]}}])
    feats = ex.map(prepare_eval_features, batched=True, remove_columns=ex.column_names)
    o = trainer.predict(feats.remove_columns(["example_id", "offset_mapping"]))
    return postprocess(ex, feats, *o.predictions)["q"]


context = ("1988년 9월 17일부터 10월 2일까지 서울특별시와 경기도 일부 수도권 지역에서 개최된 하계 올림픽이자, "
           "대회의 정식 명칭은 제24회 서울 올림픽 경기대회(Games of the XXIV Olympiad Seoul 1988)이다.")
for q in ["88올림픽 개최지는?", "88올림픽은 언제 개막했나?", "서울 올림픽의 정식 명칭은?"]:
    print(q, "→", answer(q, context))

## 7. (참고) 더 해볼 것

- `MAX_TRAIN = None`, `num_train_epochs=2`로 전체 학습하면 성능이 더 오릅니다(T4 GPU 약 1시간).
- 한국어 전용 모델 `klue/bert-base`로 `MODEL_NAME`을 바꿔 다국어 BERT와 비교해 보세요.